# 360° Construction Site Intelligence - Exploration Notebook

Demonstrates the full ML pipeline on sample panoramic imagery.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

# Load sample panorama
panorama = np.random.randint(0, 255, (512, 1024, 3), dtype=np.uint8)
print(f'Panorama shape: {panorama.shape} (H x W x C)')
print(f'Aspect ratio: {panorama.shape[1]/panorama.shape[0]:.2f}:1 (expected 2:1)')

In [ ]:
# 1. Spherical Geometry
from ml.spherical_geometry.spherical_engine import SphericalGeometryEngine

geo = SphericalGeometryEngine(cubemap_face_size=256, device='cpu')

# Extract cubemap
faces = geo.equirect_to_cubemap(panorama, face_size=256)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (face, img) in zip(axes.flat, faces.items()):
    ax.imshow(img)
    ax.set_title(str(face))
    ax.axis('off')
plt.suptitle('Cubemap Projection of 360° Panorama')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Semantic Segmentation
from ml.segmentation.panoramic_segmenter import SegFormerPanoramicSegmenter, CLASS_COLORS

segmenter = SegFormerPanoramicSegmenter(model_path=None, device='cpu', tile_size=128)
seg_result = segmenter.segment_panorama(panorama)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(panorama)
axes[0].set_title('Original Panorama')
axes[1].imshow(seg_result.colorized_mask)
axes[1].set_title(f'Semantic Segmentation (hazard score: {seg_result.hazard_score:.2f})')
axes[2].imshow(seg_result.confidence_map, cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_title('Confidence Map')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('\nTop 5 scene classes:')
for cls, area in sorted(seg_result.class_areas.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f'  {cls:<20s} {area:.1f}%')

In [ ]:
# 3. PPE Compliance
from ml.ppe.ppe_engine import PPEDetectionEngine

ppe_engine = PPEDetectionEngine(model_path=None, device='cpu', use_tracker=False)
ppe_report = ppe_engine.analyze_panorama(panorama, 'demo_panorama_001')

print(f'Workers detected: {ppe_report.total_workers}')
print(f'Compliance rate: {ppe_report.compliance_rate:.1%}')
print(f'Risk level: {ppe_report.risk_level.upper()}')
if ppe_report.alerts:
    print('Alerts:')
    for alert in ppe_report.alerts:
        print(f'  {alert}')

In [ ]:
# 4. Hazard Analysis
from ml.hazards.hazard_engine import HazardZoneEngine, colorize_risk_map

hazard_engine = HazardZoneEngine(device='cpu')
hazard_result = hazard_engine.analyze_panorama(panorama, 'demo_panorama_001', seg_result=seg_result)

risk_overlay = colorize_risk_map(hazard_result.risk_map, base_image=panorama, alpha=0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(hazard_result.risk_map, cmap='RdYlGn_r', vmin=0, vmax=1)
axes[0].set_title(f'Risk Map (overall: {hazard_result.overall_risk_score:.2f} | {hazard_result.risk_level})')
axes[1].imshow(risk_overlay)
axes[1].set_title('Risk Overlay on Panorama')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'\nHazard zones found: {len(hazard_result.zones)}')
for zone in hazard_result.zones[:3]:
    print(f'  {zone.hazard_type.value}: severity={zone.severity:.2f}, area={zone.area_percent:.1f}%')

In [ ]:
# 5. Occupancy Heatmap
from ml.occupancy.occupancy_engine import SpatialOccupancyEngine, OccupancyFrame, blend_heatmap_on_panorama

occ_engine = SpatialOccupancyEngine(heatmap_sigma=40)
worker_positions = np.random.uniform(0, 1, (8, 2)) * [panorama.shape[1], panorama.shape[0]]
worker_boxes = np.zeros((8, 4))
for i, (x, y) in enumerate(worker_positions):
    worker_boxes[i] = [x-20, y-30, x+20, y+30]

frame = OccupancyFrame(
    panorama_id='demo', timestamp=0.0,
    worker_positions=worker_positions.astype(np.float32),
    worker_ids=[f'w{i}' for i in range(8)],
    worker_boxes=worker_boxes.astype(np.float32),
)
occ_result = occ_engine.analyze_frame(frame, panorama.shape[:2], 'demo_session')

heatmap_overlay = blend_heatmap_on_panorama(panorama, occ_result.density_heatmap)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.imshow(occ_result.density_heatmap, cmap='hot')
plt.title(f'Worker Density Heatmap (utilization: {occ_result.spatial_utilization:.1%})')
plt.colorbar()
plt.subplot(1, 2, 2)
plt.imshow(heatmap_overlay)
plt.title('Heatmap Overlay')
plt.tight_layout()
plt.show()